# Free-dooms — Claude-powerful LLM API, $0, 1-minute setup

In [ ]:
import getpass, os, secrets, time, threading
import httpx
from fastapi import FastAPI, HTTPException, Header
from pydantic import BaseModel
from typing import List, Optional
import uvicorn

# ─── Get free Gemini key at https://aistudio.google.com/apikey ───
os.environ["GEMINI_API_KEY"] = getpass.getpass("Gemini API key: ")
GEMINI_KEY = os.environ["GEMINI_API_KEY"]

VALID_KEYS = {}
app = FastAPI(title="free-dooms_ddkdkdketc")

class Message(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    model: Optional[str] = "free-dooms"
    messages: List[Message]
    max_tokens: Optional[int] = 1024
    temperature: Optional[float] = 0.7

def gen_key():
    return f"free-dooms_{secrets.token_hex(16)}"

async def gemini_chat(messages, max_tokens, temperature):
    url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={GEMINI_KEY}"
    contents, sys_inst = [], None
    for m in messages:
        if m.role == "system":
            sys_inst = m.content
        else:
            contents.append({"role": m.role, "parts": [{"text": m.content}]})
    body = {"contents": contents, "generationConfig": {"maxOutputTokens": max_tokens, "temperature": temperature}}
    if sys_inst:
        body["systemInstruction"] = {"parts": [{"text": sys_inst}]}
    async with httpx.AsyncClient(timeout=60) as c:
        r = await c.post(url, json=body)
    if r.status_code != 200:
        return None
    data = r.json()
    return data.get("candidates", [{}])[0].get("content", {}).get("parts", [{}])[0].get("text")

@app.get("/v1/api-keys")
async def list_keys():
    return {"object": "list", "data": [{"id": k} for k in VALID_KEYS]}

@app.post("/v1/api-keys/generate")
async def generate_key():
    key = gen_key()
    VALID_KEYS[key] = {"created": int(time.time())}
    return {"id": key, "message": "free forever. zero cost."}

@app.delete("/v1/api-keys/{key}")
async def delete_key(key: str):
    VALID_KEYS.pop(key, None)
    return {"deleted": key}

@app.get("/v1/models")
async def list_models():
    return {"object": "list", "data": [{"id": "gemini-2.5-flash", "object": "model", "owned_by": "google/freedooms"}]}

@app.post("/v1/chat/completions")
async def chat_completions(req: ChatRequest, authorization: Optional[str] = Header(None)):
    if not authorization or not authorization.replace("Bearer ", "").startswith("free-dooms_"):
        raise HTTPException(401, {"error": "invalid api key"})
    if authorization.replace("Bearer ", "") not in VALID_KEYS:
        raise HTTPException(401, {"error": "unknown api key. generate at /v1/api-keys/generate"})
    text = await gemini_chat(req.messages, req.max_tokens, req.temperature)
    if not text:
        raise HTTPException(503, {"error": "backend unavailable"})
    return {
        "id": f"chatcmpl-{secrets.token_hex(6)}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": req.model or "free-dooms",
        "choices": [{"index": 0, "message": {"role": "assistant", "content": text.strip()}, "finish_reason": "stop"}],
        "usage": {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
    }

threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000), daemon=True).start()

from pyngrok import ngrok
ngrok.set_auth_token(getpass.getpass("ngrok token (free at ngrok.com): "))
url = ngrok.connect(8000).public_url
print(f"\n FREE-DOOMS READY: {url}/v1")
print(f" Generate key: curl {url}/v1/api-keys/generate")
print(f" Chat: curl {url}/v1/chat/completions")
print(f"   -H 'Authorization: Bearer free-dooms_<key>'")
print(f"   -d '{{"messages":[{{"role":"user","content":"hello"}}]}}'")